In [142]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import os
import yaml
import shutil

## Set the file of the original case

In [143]:
original_case = pathlib.Path(os.getcwd()).joinpath("thermstor_20year")
print(f"Original case directory: {original_case}")

Original case directory: c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\paper_runs\dual_runs\final_report_runs\thermstor_20year


## Check data files exist

In [144]:
timeseries = {
    "fuels": original_case.joinpath("Fuels_data.csv"),
    "loads": original_case.joinpath("Load_data.csv"),
    "variability": original_case.joinpath("Generators_variability.csv"),
}

singletons = {
    "generators": original_case.joinpath("Generators_data.csv"),
    "fusion": original_case.joinpath("Fusion_data.csv"),
    "co2": original_case.joinpath("CO2_cap.csv"),
    "settings": original_case.joinpath("Settings", "genx_settings.yml"),
    "networks": original_case.joinpath("Network.csv"),
}

julia_runfiles = {
    "Run": original_case.joinpath("Run.jl"),
    "Run_dual": original_case.joinpath("Run_dual.jl"),
}

tdr_dir = original_case.joinpath("tdr_setup")

# Check each file or dir exists
for name, path in timeseries.items():
    assert path.exists(), f"Missing timeseries data file: {path}"

for name, path in singletons.items():
    assert path.exists(), f"Missing single data file: {path}"

print("All required data files are present")

All required data files are present


## Load the data

In [145]:
timeseries_data = {}
for name, path in timeseries.items():
    timeseries_data[name] = pd.read_csv(path, index_col=0)
    print(f"Loaded {name} data")

# Create new index column for timeseries_data["loads"]
if timeseries_data["loads"].index.name == "Voll":
    timeseries_data["loads"].reset_index(drop=False, inplace=True)
    
singleton_data = {}
for name, path in singletons.items():
    if path.suffix == ".csv":
        singleton_data[name] = pd.read_csv(path, index_col=0)
        print(f"Loaded {name} data from CSV")
    elif path.suffix in [".yml", ".yaml"]:
        with open(path, 'r') as f:
            singleton_data[name] = yaml.safe_load(f)
        print(f"Loaded {name} data from YAML")
    else:
        print(f"Unsupported file format for {name}: {path.suffix}")

Loaded fuels data
Loaded loads data
Loaded variability data
Loaded generators data from CSV
Loaded fusion data from CSV
Loaded co2 data from CSV
Loaded settings data from YAML
Loaded networks data from CSV


In [146]:
def split_fuels_data(scenarios: list, fuels_df: pd.DataFrame, new_scenarios: list, timesteps_per_year: int):
    for i, scenario in enumerate(new_scenarios):
        segment_df = fuels_df.iloc[[0]]
        for year in scenario:
            start_row = 1 + (year - 1) * timesteps_per_year
            end_row = start_row + timesteps_per_year
            segment_df = pd.concat([segment_df, fuels_df.iloc[start_row:end_row]], ignore_index=True)
    # Give the index the header "Time_Index"
        segment_df.index.name = "Time_Index"
        scenarios[i]["fuels"] = segment_df
    return None

def split_load_data(scenarios, loads_df: pd.DataFrame, new_scenarios: list, timesteps_per_year: int):
    # Find the first column in loads_df which starts with "Load_MW_z"
    load_columns = [idx for idx, col in enumerate(loads_df.columns) if col.startswith("Load_MW_z")]
    if not load_columns:
        raise ValueError("No columns starting with 'Load_MW_z' found in Load_data.csv")
    first_load_col = load_columns[0]
    for i, scenario in enumerate(new_scenarios):
        # Create an empty dataframe with the headers of loads_df and the first row
        segment_df = loads_df.iloc[0:0].copy()
        for year in scenario:
            start_row = (year - 1) * timesteps_per_year
            end_row = start_row + timesteps_per_year
            year_data = loads_df.iloc[start_row:end_row, first_load_col:].copy()
            segment_df = pd.concat([segment_df, year_data], ignore_index=True)
        segment_df.loc[:, "Voll"] = loads_df.loc[:, "Voll"]
        segment_df.loc[:, "Demand_Segment"] = loads_df.loc[:, "Demand_Segment"]
        segment_df.loc[:, "Cost_of_Demand_Curtailment_per_MW"] = loads_df.loc[:, "Cost_of_Demand_Curtailment_per_MW"]
        segment_df.loc[:, "Max_Demand_Curtailment"] = loads_df.loc[:, "Max_Demand_Curtailment"]
        segment_df.loc[:, "Time_Index"] = range(1, len(segment_df) + 1)
        # Set Rep_Periods = new_scenario_length
        segment_df.loc[0, 'Rep_Periods'] = len(scenario)
        # Set Timesteps_per_Rep_Period = timesteps_per_year
        segment_df.loc[0, "Timesteps_per_Rep_Period"] =  timesteps_per_year
        # Set the first n entries of Sub_Weights to each with value: timesteps_per_year / timesteps_per_year, where n = new_scenario_length
        sub_weights = [timesteps_per_year / len(scenario)] * len(scenario)
        segment_df.loc[0:len(scenario)-1, "Sub_Weights"] = sub_weights
        scenarios[i]["loads"] = segment_df
    return None

def split_variability_data(scenarios, variability_df: pd.DataFrame, new_scenarios: list, timesteps_per_year: int):
    # This one is simple, just take a copy of the start:end rows for each year in the scenario
    for i, scenario in enumerate(new_scenarios):
        # segment_df = variability_df.iloc[[0]]
        for year in scenario:
            start_row = (year - 1) * timesteps_per_year
            end_row = start_row + timesteps_per_year
            if year != scenario[0]:
                segment_df = pd.concat([segment_df, variability_df.iloc[start_row:end_row]], ignore_index=True)
            else:
                segment_df = variability_df.iloc[start_row:end_row]

        # segment_df.drop(index=1, inplace=True)
        segment_df.index = segment_df.index + 1
        segment_df.index.names = ["Time_Index"]
        scenarios[i]["variability"] = segment_df
    return None

def make_period_map(scenarios: list, new_scenarios: list):
    for i, scenario in enumerate(new_scenarios):
        period_map = pd.DataFrame(columns=["Period_Index", "Rep_Period", "Rep_Period_Index"])
        for j, year in enumerate(scenario):
            period_map.loc[j] = [j + 1, year, j + 1]
        scenarios[i]["period_map"] = period_map
    return None

def adjust_settings(scenarios: list, settings: dict, new_scenarios: list):
    for i, scenario in enumerate(new_scenarios):
        new_settings = settings.copy()
        # Set CO2CapPeriods = len(scenario)
        new_settings["CO2CapPeriods"] = len(scenario)
        scenarios[i]["settings"] = new_settings
    return None

## Set the number of years to include in each new scenario segment

In [ ]:
case_name = "1year"
original_scenario_length = 20
TIMESTEPS_PER_YEAR = 8760
new_scenarios = [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10],
                     [11], [12], [13], [14], [15], [16], [17], [18], [19], [20], [4,11,15,17], [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20], [4,11,15]]
num_new_scenarios = len(new_scenarios)
scenarios = [{} for _ in range(num_new_scenarios)]

In [ ]:
split_fuels_data(
    scenarios, timeseries_data["fuels"], new_scenarios, TIMESTEPS_PER_YEAR
)
split_load_data(
    scenarios, timeseries_data["loads"], new_scenarios, TIMESTEPS_PER_YEAR
)
split_variability_data(
    scenarios, timeseries_data["variability"], new_scenarios, TIMESTEPS_PER_YEAR
)
print("Segmented timeseries data into new scenarios.")

make_period_map(scenarios, new_scenarios)
print("Created period maps for new scenarios.")

adjust_settings(scenarios, singleton_data["settings"], new_scenarios)
print("Adjusted settings for new scenarios.")

# Make a dir called "thermstor_{case_name}"
new_case_dir = pathlib.Path(os.getcwd()).joinpath(f"thermstor_{case_name}")
new_case_dir.mkdir(exist_ok=True)
print(f"Created new case directory: {new_case_dir}")
# Create a subdir called "Scenarios" in new_case_dir
scenarios_dir = new_case_dir.joinpath("Scenarios")
scenarios_dir.mkdir(exist_ok=True)

for i, scenario in enumerate(new_scenarios):
    scenario_dir = scenarios_dir.joinpath(f"Scenario_{i+1}")
    scenario_dir.mkdir(exist_ok=True)
    
    # Save fuels data
    fuels_path = scenario_dir.joinpath("Fuels_data.csv")
    scenarios[i]["fuels"].to_csv(fuels_path)
    
    # Save loads data
    loads_path = scenario_dir.joinpath("Load_data.csv")
    scenarios[i]["loads"].to_csv(loads_path, index=False)
    
    # Save variability data
    variability_path = scenario_dir.joinpath("Generators_variability.csv")
    scenarios[i]["variability"].to_csv(variability_path)

    # Save the three timeseries again in scenario_dir.joinpath("tdr_setup")/...
    tdr_setup_dir = scenario_dir.joinpath("tdr_setup")
    tdr_setup_dir.mkdir(exist_ok=True)
    fuels_tdr_path = tdr_setup_dir.joinpath("Fuels_data.csv")
    scenarios[i]["fuels"].to_csv(fuels_tdr_path)
    loads_tdr_path = tdr_setup_dir.joinpath("Load_data.csv")
    scenarios[i]["loads"].to_csv(loads_tdr_path, index=False)
    variability_tdr_path = tdr_setup_dir.joinpath("Generators_variability.csv")
    scenarios[i]["variability"].to_csv(variability_tdr_path)
    # Save period map
    period_map_path = tdr_setup_dir.joinpath("Period_Map.csv")
    scenarios[i]["period_map"].to_csv(period_map_path, index=False)
    
    # Save settings in Settings/genx_settings.yml
    settings_dir = scenario_dir.joinpath("Settings")
    settings_dir.mkdir(exist_ok=True)
    settings_path = settings_dir.joinpath("genx_settings.yml")
    with open(settings_path, 'w') as f:
        yaml.dump(scenarios[i]["settings"], f)
    # Copy the Settings/gurobi_settings.yml from original_case to scenario_dir/Settings using shutil.copyfile
    gurobi_src_path = original_case.joinpath("Settings", "gurobi_settings.yml")
    gurobi_dst_path = settings_dir.joinpath("gurobi_settings.yml")
    shutil.copyfile(gurobi_src_path, gurobi_dst_path)

    # Copy singleton data files to scenario_dir
    for name, path in singletons.items():
        if name != "settings":  # settings already handled
            dst_path = scenario_dir.joinpath(path.name)
            shutil.copyfile(path, dst_path)

    # Copy Julia run files to scenario_dir
    for name, path in julia_runfiles.items():
        dst_path = scenario_dir.joinpath(path.name)
        shutil.copyfile(path, dst_path)
        
        
        modified_lines = []
        with open(dst_path, 'r') as file:
            lines = file.readlines()
            for line in lines:
                modified_lines.append(line.replace("for year_num in 1:20", f"for year_num in 1:{len(scenario)}").replace("num_years = 20", f"num_years = {len(scenario)}"))
        with open(dst_path, 'w') as file:
            file.writelines(modified_lines)
    print(f"Saved Scenario {i+1} data to {scenario_dir}")

Segmented timeseries data into new scenarios.
Created period maps for new scenarios.
Adjusted settings for new scenarios.
Created new case directory: c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\paper_runs\dual_runs\final_report_runs\thermstor_1year
Saved Scenario 1 data to c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\paper_runs\dual_runs\final_report_runs\thermstor_1year\Scenarios\Scenario_1
Saved Scenario 2 data to c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\paper_runs\dual_runs\final_report_runs\thermstor_1year\Scenarios\Scenario_2
Saved Scenario 3 data to c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\paper_runs\dual_runs\final_report_runs\thermstor_1year\Scenarios\Scenario_3
Saved Scenario 4 data to c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\paper_runs\dual_runs\final_report_runs\thermstor_1year\Scenarios\Scenario_4
Saved Scenario 5 data to c:\Users\INVITE\Desktop\GenX-fusion\GenX-fusion\fusion_paper\